# 🌿 Plant Disease Identification & Health Diagnosis System
### Deep Learning Convolutional Neural Network (CNN) for Multi-Class Leaf Disease Classification

--- 

## 📌 Executive Summary & Problem Statement
Agricultural crop diseases significantly impact global food security, crop yield, and farm profitability. Early and accurate detection of plant foliar diseases is critical for timely agricultural intervention. 

In this notebook, we develop an end-to-end Computer Vision system using a custom **Deep Convolutional Neural Network (CNN)** trained on the **PlantVillage** dataset. The dataset comprises over 20,000 leaf images categorized into **15 distinct health & disease classes** across three major crops:
- 🫑 **Pepper (Bell):** Bacterial Spot, Healthy
- 🥔 **Potato:** Early Blight, Late Blight, Healthy
- 🍅 **Tomato:** Bacterial Spot, Early Blight, Late Blight, Leaf Mold, Septoria Leaf Spot, Spider Mites, Target Spot, Yellow Leaf Curl Virus, Mosaic Virus, Healthy

### Workflow Overview:
1. **Environment Setup & Data Ingestion**: Load images, verify dataset integrity, index classes.
2. **Exploratory Data Analysis (EDA)**: Class distribution visualization and sample inspection.
3. **Preprocessing & Augmentation**: Stratified 80/10/10 split, spatial resizing ($128\times 128$), rotation/flip augmentations, and ImageNet tensor normalization.
4. **Custom CNN Architecture (`PlantDiseaseCNN`)**: 4 Convolutional blocks (Conv2D -> BatchNorm -> ReLU -> MaxPool -> Dropout) + Global Adaptive Average Pooling + Dense Classifier.
5. **Training & Optimization**: Train using AdamW optimizer with learning rate scheduler (`ReduceLROnPlateau`) and CrossEntropyLoss.
6. **Model Evaluation & Visualizations**: Overall Test Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and Prediction Grids.
7. **Model Deployment Artifacts**: Export weights (`plant_disease_cnn.pth`) and metadata (`class_names.json`) for Streamlit/Gradio web apps.

## Step 1: Environment Setup & Library Imports

In [ ]:
import os
import sys
import json
import time
from glob import glob
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

from src.dataset import prepare_dataloaders, scan_plant_village
from src.model import PlantDiseaseCNN
from src.utils import DISEASE_INFO, clean_class_name

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version : {torch.__version__}')
print(f'Active Computing Device : {device}')

## Step 2: Dataset Scanning & Exploratory Data Analysis (EDA)
We inspect the dataset structure, count the total images per class, and visualize sample leaf images from each category.

In [ ]:
data_dir = r'c:\Users\cnive\Ai training\CNN\PlantVillage'
class_names, image_paths, labels = scan_plant_village(data_dir)

# Create DataFrame for EDA
df_eda = pd.DataFrame({
    'image_path': image_paths,
    'raw_class': [class_names[i] for i in labels],
    'clean_name': [clean_class_name(class_names[i]) for i in labels]
})

print(f'Total Images Found : {len(df_eda)}')
print(f'Total Classes      : {len(class_names)}')
df_summary = df_eda['clean_name'].value_counts().reset_index()
df_summary.columns = ['Disease / Health Category', 'Image Count']
display(df_summary)

In [ ]:
# Visualize Class Distribution
plt.figure(figsize=(12, 6))
sns.barplot(data=df_summary, x='Image Count', y='Disease / Health Category', palette='viridis')
plt.title('PlantVillage Dataset Class Distribution (15 Categories)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Images')
plt.ylabel('')
for i, v in enumerate(df_summary['Image Count']):
    plt.text(v + 30, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Sample Image Visualizations Grid
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for idx, cls in enumerate(class_names):
    sample_path = df_eda[df_eda['raw_class'] == cls]['image_path'].iloc[0]
    img = Image.open(sample_path)
    axes[idx].imshow(img)
    axes[idx].set_title(clean_class_name(cls), fontsize=9, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle('Sample Leaf Images Across 15 Categories', fontsize=16, fontweight='bold', y=1.02)
plt.show()

## Step 3: Data Preprocessing & Augmentation Pipelines
We split data into **Train (80%)**, **Validation (10%)**, and **Test (10%)** sets using stratified sampling to maintain identical class ratios. Data augmentation (rotations, flips, color jitter) is applied exclusively to the training set.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 64

train_loader, val_loader, test_loader, split_info = prepare_dataloaders(
    data_dir, img_size=IMG_SIZE, batch_size=BATCH_SIZE
)

print(f'Train Batches      : {len(train_loader)} ({split_info["train_size"]} images)')
print(f'Validation Batches : {len(val_loader)} ({split_info["val_size"]} images)')
print(f'Test Batches       : {len(test_loader)} ({split_info["test_size"]} images)')

## Step 4: Deep CNN Model Architecture Definition
We construct `PlantDiseaseCNN` featuring 4 Convolutional blocks, Batch Normalization, ReLU activations, Max Pooling, Dropout regularization, Global Adaptive Pooling, and a Dense classification head.

In [ ]:
model = PlantDiseaseCNN(num_classes=len(class_names), dropout_rate=0.4).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal Parameters    : {total_params:,}')
print(f'Trainable Parameters : {trainable_params:,}')

## Step 5: Model Training & Validation Loop

In [ ]:
# Training configuration
EPOCHS = 6
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

MODEL_SAVE_PATH = 'plant_disease_cnn.pth'
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('Starting Model Training...')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # Training step
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        t_correct += (preds == lbls).sum().item()
        t_total += lbls.size(0)
    
    train_loss = t_loss / t_total
    train_acc = t_correct / t_total
    
    # Validation step
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            loss = criterion(outs, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outs, 1)
            v_correct += (preds == lbls).sum().item()
            v_total += lbls.size(0)
            
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    epoch_time = time.time() - epoch_start
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.1f}s) | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')
          
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'  --> Saved best weights checkpoint (Val Acc: {val_acc*100:.2f}%)')

print(f'Total Training Time: {(time.time() - start_time)/60:.2f} mins')

## Step 6: Model Evaluation & Performance Metrics
We evaluate the saved best model on the unseen **Test Dataset**, calculating accuracy, precision, recall, F1-score, and generating confusion matrix heatmaps.

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

y_pred, y_true = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        outs = model(imgs)
        _, preds = torch.max(outs, 1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(lbls.cpu().numpy())

test_acc = accuracy_score(y_true, y_pred)
clean_labels = [clean_class_name(c) for c in class_names]

print(f'=== OVERALL TEST ACCURACY: {test_acc*100:.2f}% ===\n')
print(classification_report(y_true, y_pred, target_names=clean_labels, digits=4))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clean_labels, yticklabels=clean_labels)
plt.title('Plant Disease Classification - Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Sample Test Prediction Grid
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

sample_indices = np.random.choice(len(test_loader.dataset), 12, replace=False)
for i, idx in enumerate(sample_indices):
    img_path = test_loader.dataset.image_paths[idx]
    true_lbl = test_loader.dataset.labels[idx]
    
    raw_img = Image.open(img_path).convert('RGB')
    img_tensor = eval_transform(raw_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(img_tensor)
        prob = torch.softmax(out, dim=1)
        pred_lbl = int(prob.argmax())
        conf = float(prob.max()) * 100
        
    color = 'green' if pred_lbl == true_lbl else 'red'
    axes[i].imshow(raw_img)
    title_text = f'True: {clean_class_name(class_names[true_lbl])}\nPred: {clean_class_name(class_names[pred_lbl])} ({conf:.1f}%)'
    axes[i].set_title(title_text, fontsize=9, color=color, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Sample Predictions on Test Images', fontsize=16, fontweight='bold', y=1.02)
plt.show()

## Step 7: Saved Artifacts & Web App Deployment
The model has been successfully trained and evaluated. Artifacts created for deployment:
1. `plant_disease_cnn.pth`: Trained PyTorch CNN model weights.
2. `class_names.json`: Category index mappings.
3. `app.py`: Interactive Streamlit Web Application (`streamlit run app.py`).
4. `gradio_app.py`: Interactive Gradio Web Application (`python gradio_app.py`).